In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if not candidate.suffix and (candidate.is_absolute() or ':' in text):
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from bisect import bisect_left
import pickle


In [ ]:
gene_exp = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Expression_Trimmed.csv')
gene_exp.name = 'gene_exp'

copy_num = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Copy_Number_Trimmed.csv')
copy_num.name = 'copy_num'

shRNA = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'shRNA_Trimmed.csv')
shRNA.name = 'shRNA'

gene_mut = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Mutation_Trimmed.csv')
gene_mut.name = 'gene_mut'

CRISPR = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'CRISPR_Trimmed.csv')
CRISPR.name = 'CRISPR'


In [ ]:
biogrid_pairs = (
    pd.read_csv(DATA_DIR / 'clean_biogrid_interactions.csv')
    .rename(columns={'Gene_A': 'Gene1', 'Gene_B': 'Gene2'})
)
biogrid_pairs['Gene1'] = biogrid_pairs['Gene1'].astype(str).str.strip()
biogrid_pairs['Gene2'] = biogrid_pairs['Gene2'].astype(str).str.strip()
biogrid_pairs = biogrid_pairs.dropna(subset=['Gene1', 'Gene2']).reset_index(drop=True)


In [ ]:
genes = sorted(set(gene_exp.iloc[:,0].str.strip()) & 
               set(copy_num.iloc[:,0].str.strip()) & 
               set(shRNA.iloc[:,0].str.strip()) & 
               set(CRISPR.iloc[:,0].str.strip()))
genes = [i.strip() for i in genes]


In [ ]:
datasets = gene_exp, copy_num, shRNA, gene_mut, CRISPR
points = density_centers(gene_exp, 7), [0,1,2,3,4,6,8], density_centers(shRNA, 7), [0, 1], density_centers(CRISPR, 7)
cont = True, False, True, False, True


In [ ]:
a = ['CCND1.CCNE1', 'CCNA2.CCND1', 'CCNB1.CCND1', 'CCNA2.CCNE1', 'CCNB1.CCNE1', 'CDK1.CDK4', 'CDK1.CDK6', 'CDK2.CDK4', 'CCNB1.CDK4', 'CCNB1.CDK6', 'RB1.RBL1', 'E2F2.E2F3', 'E2F2.HDAC1', 'E2F2.RBL1', 'E2F3.HDAC1', 'MCM3.ORC1', 'MCM4.ORC1', 'MCM5.ORC1', 'MCM6.ORC1', 'CDT1.ORC1', 'MCM4.ORC2', 'MCM6.ORC2', 'MCM3.ORC3', 'MCM4.ORC3', 'MCM5.ORC3', 'MCM6.ORC3', 'CDT1.ORC3', 'MCM2.ORC4', 'MCM4.ORC4', 'MCM5.ORC4', 'MCM6.ORC4', 'MCM7.ORC4', 'CDT1.ORC4', 'MCM4.ORC5', 'MCM5.ORC5', 'MCM6.ORC5', 'MCM7.ORC5', 'CDT1.ORC5', 'MCM3.ORC6', 'MCM5.ORC6', 'MCM6.ORC6', 'MCM7.ORC6', 'CDT1.ORC6', 'CDT1.MCM3', 'CDC6.MCM4', 'CDT1.MCM5', 'CDC6.MCM6', 'CDT1.MCM7', 'ANAPC1.ANAPC3', 'ANAPC1.ANAPC6', 'ANAPC1.ANAPC8', 'ANAPC1.ANAPC9', 'ANAPC1.ANAPC11', 'ANAPC1.CDH1', 'ANAPC2.ANAPC3', 'ANAPC2.ANAPC6', 'ANAPC2.ANAPC8', 'ANAPC2.ANAPC9', 'ANAPC2.CDH1', 'ANAPC3.ANAPC4', 'ANAPC3.ANAPC5', 'ANAPC3.ANAPC6', 'ANAPC3.ANAPC7', 'ANAPC3.ANAPC8', 'ANAPC3.ANAPC9', 'ANAPC10.ANAPC3', 'ANAPC11.ANAPC3', 'ANAPC3.CDC20', 'ANAPC3.CDH1', 'ANAPC4.ANAPC6', 'ANAPC4.ANAPC8', 'ANAPC4.ANAPC9', 'ANAPC4.CDH1', 'ANAPC5.ANAPC6', 'ANAPC5.ANAPC8', 'ANAPC5.ANAPC9', 'ANAPC11.ANAPC5', 'ANAPC5.CDH1', 'ANAPC6.ANAPC7', 'ANAPC6.ANAPC8', 'ANAPC6.ANAPC9', 'ANAPC10.ANAPC6', 'ANAPC11.ANAPC6', 'ANAPC6.CDC20', 'ANAPC6.CDH1', 'ANAPC7.ANAPC8', 'ANAPC7.ANAPC9', 'ANAPC11.ANAPC7', 'ANAPC7.CDH1', 'ANAPC8.ANAPC9', 'ANAPC10.ANAPC8', 'ANAPC11.ANAPC8', 'ANAPC8.CDC20', 'ANAPC8.CDH1', 'ANAPC10.ANAPC9', 'ANAPC11.ANAPC9', 'ANAPC9.CDC20', 'ANAPC9.CDH1', 'ANAPC10.CDH1', 'ANAPC11.CDH1', 'MRE11.RAD50', 'MRE11.NBN', 'BARD1.PALB2', 'FANCB.FANCE', 'FANCB.FANCD2', 'FANCE.FANCL', 'FANCD2.FANCF', 'ATR.ATRIP', 'ATRIP.CHEK1', 'ATRIP.TOPBP1', 'ATRIP.CLSPN', 'CHEK1.TOPBP1', 'CLSPN.TOPBP1', 'PIK3CA.PIK3CB', 'AKT2.PIK3CA', 'MTOR.PIK3CA', 'PIK3CA.RICTOR', 'PIK3CA.RPTOR', 'MLST8.PIK3CA', 'PIK3CA.TSC1', 'PIK3CA.TSC2', 'AKT1.PIK3CB', 'AKT2.PIK3CB', 'MTOR.PIK3CB', 'PIK3CB.RICTOR', 'PIK3CB.RPTOR', 'MLST8.PIK3CB', 'PIK3CB.TSC1', 'PIK3CB.TSC2', 'AKT1.AKT2', 'AKT1.RPTOR', 'AKT1.MLST8', 'AKT1.TSC2', 'AKT2.MTOR', 'AKT2.RICTOR', 'AKT2.RPTOR', 'AKT2.MLST8', 'AKT2.TSC1', 'AKT2.TSC2', 'MTOR.TSC1', 'MTOR.TSC2', 'RICTOR.TSC1', 'RICTOR.TSC2', 'RPTOR.TSC1', 'RPTOR.TSC2', 'MLST8.TSC1', 'MLST8.TSC2', 'KRAS.MAPK1', 'MAP2K1.NRAS', 'MAPK1.NRAS', 'PRKAA1.PRKAA2', 'BCL2.MCL1', 'BCL2L1.MCL1', 'BAX.BCL2L11', 'BAK1.BID', 'BAK1.BCL2L11', 'BCL2L11.BID', 'CASP9.CYCS', 'IL1B.NLRP3', 'IL1B.PYCARD', 'ATG13.ULK1', 'PIK3C3.ULK1', 'ATG5.ULK1', 'ATG7.ULK1', 'ATG13.RB1CC1', 'ATG13.BECN1', 'ATG13.PIK3C3', 'ATG13.ATG5', 'ATG13.ATG7', 'BECN1.RB1CC1', 'PIK3C3.RB1CC1', 'ATG7.RB1CC1', 'ATG5.BECN1', 'ATG7.BECN1', 'ATG5.PIK3C3', 'ATG7.PIK3C3', 'ARID1A.BRD9', 'ARID1B.PBRM1', 'ARID1B.BRD9', 'BRD9.SMARCB1', 'BRD9.PBRM1', 'EPC1.EPC2', 'ATM.MDM4', 'CHEK2.MDM4', 'MDM4.SFN', 'ATM.SFN', 'CHEK2.SFN', 'PSMA1.PSMD9', 'PSMA1.PSMD10', 'PSMA2.PSMD9', 'PSMA2.PSMD10', 'PSMA3.PSMD9', 'PSMA3.PSMD10', 'PSMA4.PSMD9', 'PSMA4.PSMD10', 'PSMA5.PSMD9', 'PSMA5.PSMD10', 'PSMA6.PSMD9', 'PSMA6.PSMD10', 'PSMA7.PSMD9', 'PSMA7.PSMD10', 'PSMB1.PSMD9', 'PSMB1.PSMD10', 'PSMB2.PSMD9', 'PSMB2.PSMD10', 'PSMB3.PSMD9', 'PSMB3.PSMD10', 'PSMB4.PSMD10', 'PSMB5.PSMD9', 'PSMB5.PSMD10', 'PSMB6.PSMD9', 'PSMB6.PSMD10', 'PSMB7.PSMD9', 'PSMB7.PSMD10', 'PSMC1.PSMD9', 'HSP90AA1.STIP1', 'HSP90AB1.STIP1', 'CDC37.STIP1', 'ATF4.EIF2S1', 'EIF2AK4.EIF2S1', 'DDIT3.EIF2S1', 'ATF4.EIF2AK4', 'DDIT3.EIF2AK4']
x = []
y = []
for pair in a:
    g1, g2 = pair.split('.')
    x.append(g1)
    y.append(g2)

biogrid_pairs = pd.DataFrame({'Gene1' : x, 'Gene2' : y})
biogrid_pairs

In [ ]:
temp_time = time.time()
pair_mask = biogrid_pairs['Gene1'].isin(genes) & biogrid_pairs['Gene2'].isin(genes)
pair_table = biogrid_pairs.loc[pair_mask].reset_index(drop=True)
pair_matrix = build_density_map(
    datasets,
    pair_table[['Gene1', 'Gene2']].to_numpy().tolist(),
    points,
    cont,
)
pair_matrix = pd.concat([pair_table, pair_matrix], axis=1)
print(f"converted {len(pair_table)} pairs in {time.time() - temp_time:.2f}s")


In [ ]:
output_path = ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_missing.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
pair_matrix.to_csv(output_path, index=False)
output_path


In [ ]:
df = pd.concat([pd.read_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm.csv'), 
                pd.read_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_missing.csv')], axis=0, ignore_index=True)
df

In [ ]:
df.to_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_combined.csv', index=False)

In [ ]:
pair_matrix.head()


In [ ]:
pair_matrix.shape


In [ ]:
with open(ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl', 'rb') as file:
    pos = pickle.load(file)
list(pd.DataFrame(pos[0]).iloc[1])